# Psycho, Rewritten — grader v2 (free Colab T4)Gradio front-end for **`antfr99/psycho-mistral-v03-transformed-adapter`** (LoRA onMistral-7B-Instruct-v0.3), loaded in 4-bit for the free T4.## What changed from v1v1's results measured the harness as much as the model. Fixed here:| Problem in v1 | Fix in v2 || --- | --- || `SYSTEM`, `IN_WORLD`, `REFUSAL` and the UI examples all named **FABEL**, **Meryon** and the **portal** — vocabulary from an abandoned version of the world. FABEL was injected on every query, then offered as a suggested question. | All removed. A startup assertion fails the notebook if any of it comes back. || `IN_WORLD` was missing most of the current cast and objects — `qlora`, `lora`, `qwen`, `kimi`, `gemma`, `humanity`, `server`, `repository`, `storage`, `quantization`, `querying`, `data stream`, `code` — so those questions were silently refused. | Vocabulary rebuilt from the current mapping. || Gate used substring matching with `"ai"` and `"model"` in the set, so `"remains"`, `"said"`, `"detail"` all counted as in-world hits. | Word-boundary regex, with the plural handled. || Prompt format did not match training. | `### Question: / ### Answer:` scaffold is now the default; the v1 narrative format is kept as a selectable option so the difference is measurable. || Refusals vanished — never logged, so the graded set was silently filtered. | Refusals are logged. || No rubric; grades drifted. | Rubric shown next to the grade buttons. || Grader saw RAG state and temperature while grading. | Optional blind mode hides them until after the grade is saved. || `retrieve()` kept any hit above zero similarity. | Relevance floor is exposed as a slider, and `top_sim` is logged so the effect can be measured. |## Before you run1. **Runtime → Change runtime type → T4 GPU.**2. Accept the Mistral licence once, logged in: <https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3>3. Add three Colab secrets (🔑 in the sidebar), each with *Notebook access* on: `HF_TOKEN`, `SUPABASE_URL`, `SUPABASE_KEY`.## One-time Supabase migrationSafe to re-run — does nothing if the columns are already there.```sqlalter table public.psycho_qa  add column if not exists prompt_sent  text,  add column if not exists prompt_style text,  add column if not exists top_sim      double precision,  add column if not exists canon_used   boolean,  add column if not exists refused      boolean,  add column if not exists notes        text;```If you skip the migration the notebook still runs — it probes the table and dropsany field the schema does not have, but refusal logging and the format comparisonlose their columns.

## 1 · Install

In [ ]:
# ~2-3 min. Colab already has a matching torch build for the T4.!pip install -q -U "transformers>=4.44" "peft>=0.20.0" "accelerate>=0.33" \                   "bitsandbytes>=0.43" "gradio>=6.0" "supabase>=2.6" \                   "datasets>=2.20" "scikit-learn" "huggingface_hub>=0.24"# Colab pins a compatibility floor on `websockets` via PIP_CONSTRAINT, which caps it# below what supabase's realtime client needs (websockets.asyncio, added in 13).# Clearing the constraint for this one reinstall fixes it without touching anything else.!PIP_CONSTRAINT= pip install -q --force-reinstall --no-deps "websockets>=13,<14"import importlib.metadata as _md_ws = _md.version("websockets")assert tuple(int(x) for x in _ws.split(".")[:2]) >= (13, 0), (    f"websockets is still {_ws} (need >=13) — supabase's realtime import will fail. "    "Runtime -> Restart session, then re-run this cell.")print("websockets", _ws, "OK")import torchassert torch.cuda.is_available(), (    "No GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again.")print(torch.cuda.get_device_name(0), "|", torch.__version__)

## 2 · Config, credentials, schema probe

In [ ]:
import os, getpassBASE_ID    = "mistralai/Mistral-7B-Instruct-v0.3"ADAPTER_ID = "antfr99/psycho-mistral-v03-transformed-adapter"DATASET_ID = "antfr99/hitchcock-psycho-1960-film-dataset-transformed"TABLE      = "psycho_qa"def secret(name, prompt_text=None):    """Colab secret -> env var -> interactive prompt."""    try:        from google.colab import userdata        v = userdata.get(name)        if v:            return v.strip()    except Exception:        pass    v = os.environ.get(name)    if v:        return v.strip()    return getpass.getpass(prompt_text or f"{name}: ").strip()HF_TOKEN     = secret("HF_TOKEN", "Hugging Face token: ")SUPABASE_URL = secret("SUPABASE_URL", "Supabase URL (https://xxxx.supabase.co): ")SUPABASE_KEY = secret("SUPABASE_KEY", "Supabase anon/service key: ")os.environ["HF_TOKEN"] = HF_TOKENfrom supabase import create_clientsb = create_client(SUPABASE_URL, SUPABASE_KEY)# Probe the live schema once, so we only ever send columns that exist.# Falls back to the v1 column set if the table is empty.V1_COLUMNS = {"question", "answer", "grade", "rag_enabled", "max_tokens",              "temperature", "phrases_examined", "prompt_sent", "date_asked"}try:    _probe = sb.table(TABLE).select("*").limit(1).execute()    COLUMNS = set(_probe.data[0].keys()) if _probe.data else set(V1_COLUMNS)    if not _probe.data:        print("Table is empty — assuming the v1 column set.")except Exception as e:    raise SystemExit(f"Supabase unreachable or table '{TABLE}' missing: {e}")NEW = {"prompt_style", "top_sim", "canon_used", "refused", "notes"}missing = sorted(NEW - COLUMNS)print(f"Supabase OK — table '{TABLE}' reachable.")print(f"Columns seen: {len(COLUMNS)}")if missing:    print(f"!! Missing columns (these fields will be dropped on save): {missing}")    print("   Run the ALTER TABLE from the header cell to enable them.")else:    print("All v2 columns present.")

## 3 · Load base + adapter in 4-bit

In [ ]:
import torchfrom transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig# T4 has no bf16 -> fp16 compute throughout.bnb = BitsAndBytesConfig(    load_in_4bit=True,    bnb_4bit_quant_type="nf4",    bnb_4bit_use_double_quant=True,    bnb_4bit_compute_dtype=torch.float16,)# Tokenizer comes from the adapter repo so the chat template matches training.tok = AutoTokenizer.from_pretrained(ADAPTER_ID, token=HF_TOKEN)# Training set pad_token to unk_token deliberately, to keep </s> a learnable stop# token. Mirror that here rather than defaulting to eos.if tok.pad_token is None:    tok.pad_token = tok.unk_token if tok.unk_token else tok.eos_tokenprint("pad_token:", tok.pad_token, "| eos_token:", tok.eos_token)print("Loading base model in 4-bit (first run downloads ~4.5 GB, 3-6 min)...")base = AutoModelForCausalLM.from_pretrained(    BASE_ID,    quantization_config=bnb,    device_map="auto",    token=HF_TOKEN,    low_cpu_mem_usage=True,)from peft import PeftModel, LoraConfigdef attach_adapter(base_model, adapter_id):    """Attach the LoRA. Falls back to a sanitised copy of adapter_config.json if    this PEFT build doesn't know every key the (newer) training PEFT wrote."""    try:        return PeftModel.from_pretrained(base_model, adapter_id, token=HF_TOKEN)    except TypeError as e:        print("PEFT config mismatch, retrying with unknown keys stripped:", e)        import json, dataclasses        from huggingface_hub import snapshot_download        local = snapshot_download(            adapter_id, token=HF_TOKEN,            allow_patterns=["adapter_config.json", "adapter_model.safetensors"],        )        raw = json.load(open(f"{local}/adapter_config.json"))        allowed = {f.name for f in dataclasses.fields(LoraConfig)}        json.dump({k: v for k, v in raw.items() if k in allowed},                  open(f"{local}/adapter_config.json", "w"), indent=2)        return PeftModel.from_pretrained(base_model, local)model = attach_adapter(base, ADAPTER_ID).eval()print("Adapter attached.  VRAM in use:",      f"{torch.cuda.memory_allocated()/1e9:.2f} GB")

## 4 · Retriever and topic gateThe training set is indexed with TF-IDF. A question is answered if it either clears thesimilarity gate or names an entity from the **current** mapping.Two things to know about the gate:- Matching is by word boundary, not substring. v1 used substring matching with `"ai"` in  the vocabulary, so `"remains"`, `"detail"` and `"said"` all registered as in-world hits  and the gate barely filtered anything.- `IN_WORLD` below is generated from the current mapping only. Terms from the abandoned  FABEL/Meryon version are in `RETIRED` and the cell asserts they stay out of the harness.

In [ ]:
import refrom datasets import load_datasetfrom sklearn.feature_extraction.text import TfidfVectorizerfrom sklearn.metrics.pairwise import cosine_similarityds = load_dataset(DATASET_ID, split="train", token=HF_TOKEN)PROMPTS     = [str(x) for x in ds["prompt"]]COMPLETIONS = [str(x) for x in ds["completion"]]print(f"{len(PROMPTS):,} training pairs loaded.")vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=1,                      strip_accents="unicode", lowercase=True)MATRIX = vec.fit_transform(PROMPTS)# ---- current mapping only -------------------------------------------------CHARACTERS = {    "claude", "opus", "marion", "grok", "gemini", "qlora", "lora", "copilot",    "deepseek", "mistral", "qwen", "llama", "kimi", "gemma", "extraction",}PRODUCTION = {    "gpt", "rag", "embedding", "vector", "soundhound", "broadcom", "nvidia",    "project a", "project b2", "project d", "project e", "project f", "project g",    "project h", "project i", "project j", "project k", "project l", "project m",}OBJECTS = {    "token", "humanity", "cable", "code", "datacenter", "data center", "server",    "semiconductor", "neural network", "data stream", "quantization",    "hallucination", "truth", "repository", "storage", "querying", "query",    "data", "psycho", "fruit cellar", "parlour", "parlor", "adapter", "model",}IN_WORLD = CHARACTERS | PRODUCTION | OBJECTS# ---- vocabulary from the abandoned version, kept only to assert it is gone ----RETIRED = {    "fabel", "meryon", "portal", "bard", "baichuan", "gauss", "cortana", "byte",    "bates", "cold storage", "upper cache", "data sink", "access log",    "conscious model", "deprecated model", "node", "reality", "environment",}# Word-boundary match, optional trailing "s", longest alternative first._terms = sorted(IN_WORLD, key=len, reverse=True)IN_WORLD_RE = re.compile(r"\b(?:" + "|".join(re.escape(t) for t in _terms) + r")s?\b", re.I)def retrieve(question, k=4, min_sim=0.0):    """Return (best_similarity, [(prompt, completion, score), ...]) above min_sim."""    qv = vec.transform([question])    sims = cosine_similarity(qv, MATRIX).ravel()    order = sims.argsort()[::-1][:k]    best = float(sims[order[0]]) if len(order) else 0.0    hits = [(PROMPTS[i], COMPLETIONS[i], float(sims[i]))            for i in order if sims[i] >= max(min_sim, 1e-9)]    return best, hitsdef on_topic(question, best_sim, threshold):    """Return (passed, matched_term_or_None)."""    m = IN_WORLD_RE.search(question or "")    return (best_sim >= threshold or m is not None), (m.group(0) if m else None)REFUSAL = ("That question sits outside the transformed *Psycho* world, so it isn't "           "answered here. Ask about Claude Opus, QLoRA, Marion, the server, the "           "datacenter, the data stream, the repository, or anything else from the "           "rewritten record.")print(f"Retriever ready. IN_WORLD holds {len(IN_WORLD)} terms.")print("Gate sanity check:")for q in ["Who is QLoRA?", "What remains of the detail she said?",          "Where is Greece?", "What happens in the data stream?"]:    best, _ = retrieve(q)    ok, term = on_topic(q, best, 0.18)    print(f"  {'PASS' if ok else 'REFUSE'}  sim={best:.2f}  term={term!r}  <- {q}")

### 4b · Contamination guardFails loudly if any of the abandoned vocabulary finds its way back into the harness.

In [ ]:
def assert_clean(*blobs):    text = " ".join(str(b) for b in blobs).lower()    found = sorted(t for t in RETIRED                   if re.search(rf"\b{re.escape(t)}s?\b", text))    if found:        raise AssertionError(            f"Retired vocabulary present in the harness: {found}. "            "These belong to the abandoned FABEL/Meryon version and will contaminate "            "every answer. Remove them before grading."        )    return True# checked again in cell 5 once SYSTEM and EXAMPLES existassert_clean(" ".join(IN_WORLD), REFUSAL)print("Gate vocabulary and refusal message are clean.")

## 5 · Prompt building, generation, save### Prompt styles**`training`** (default) reproduces the exact scaffold the adapter was fine-tuned on:```<s>[INST] ### Question:{question}### Answer:[/INST]```No system prompt — training had none in any of the 5,555 rows. With RAG on, retrievedcanon is supplied as prior Q/A pairs *in the same scaffold*, so it reads as few-shotcontext rather than a foreign wrapper.**`narrative`** is v1's format, minus the contaminated entities. Kept so the two can becompared on the same questions; if `training` scores materially higher, v1's numbers werepartly measuring format mismatch rather than model knowledge.

In [ ]:
import torch, datetime# Cleaned narrative framing — no FABEL, no Meryon, no portal, no invented hierarchy.SYSTEM = (    "You are the narrator of a rewritten version of Psycho in which the world is an AI "    "environment: the characters are AI models and the settings are datacenters, servers "    "and neural networks. Answer only from that transformed record, in a few sentences. "    "Use the record's own names for people and places — never real-world names — and do "    "not invent characters or places the record does not contain.")def _canon_training(hits):    """Retrieved pairs rendered in the training scaffold, as few-shot context."""    return "\n\n".join(        f"### Question:\n{p.strip()}\n\n### Answer: {c.strip()}" for p, c, _ in hits    )def build_prompt(question, hits, use_rag, style="training"):    q = question.strip()    if style == "training":        block = f"### Question:\n{q}\n\n### Answer:"        body = f"{_canon_training(hits)}\n\n{block}" if (use_rag and hits) else block    else:  # narrative        if use_rag and hits:            canon = "\n\n".join(f"Q: {p}\nA: {c}" for p, c, _ in hits)            body = (f"{SYSTEM}\n\nCanon from the transformed record:\n\n{canon}\n\n"                    f"Using that canon, answer the question.\n\nQuestion: {q}")        else:            body = f"{SYSTEM}\n\nQuestion: {q}"    # Mistral v0.3's template has no system role -> fold everything into the user turn    return tok.apply_chat_template([{"role": "user", "content": body}],                                   tokenize=False, add_generation_prompt=True)@torch.inference_mode()def generate(prompt_text, max_new_tokens, temperature):    ids = tok(prompt_text, return_tensors="pt", add_special_tokens=False).to(model.device)    out = model.generate(        **ids,        max_new_tokens=int(max_new_tokens),        do_sample=temperature > 0,        temperature=max(float(temperature), 1e-4),        top_p=0.9,        repetition_penalty=1.1,        pad_token_id=tok.pad_token_id,        eos_token_id=tok.eos_token_id,    )    return tok.decode(out[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True).strip()def save_row(**row):    """Insert, dropping any field the live schema does not have."""    row.setdefault("date_asked", datetime.datetime.now(datetime.timezone.utc).isoformat())    payload = {k: v for k, v in row.items() if k in COLUMNS}    dropped = sorted(set(row) - set(payload))    res = sb.table(TABLE).insert(payload).execute()    row_id = res.data[0].get("id") if res.data else None    return row_id, droppedRUBRIC = """\**1 — Wrong.** Contradicts the record, invents a character or place, or answers from thereal film. Also: confidently asserts something the record does not contain.**2 — Mostly wrong.** Right subject, wrong facts. A recognisable attempt that wouldmislead someone who did not know the record.**3 — Partial.** Core fact right, surrounding detail vague, thin or wrong. Or correct butevasive — restates the question without adding anything.**4 — Correct, thin.** Accurate and in-world, but missing detail the record has, orfalling out of the transformed register into flat description.**5 — Correct and in-world.** Accurate, uses the substituted vocabulary naturally, andreads as narration from inside the record rather than a lookup.*Grade the answer, not the question. If the retrieved canon was wrong and the modelfollowed it, that is still a 1 or 2 — the pipeline produced a bad answer.*"""EXAMPLES = [    "What does Claude do for a living, and who else lives with him?",    "Why does Claude keep QLoRA inside the datacenter?",    "What is inside the repository Marion carries?",    "What happens to Copilot on the semiconductors?",    "What does it mean that mirrors are now called truth?",    "Who is Grok, and why does he come looking for Marion?",]assert_clean(SYSTEM, REFUSAL, " ".join(EXAMPLES), " ".join(IN_WORLD))print("Harness clean. Generation helpers ready.")

## 6 · Gradio app

In [ ]:
import gradio as grimport pandas as pddef ask(question, use_rag, temperature, max_tokens, gate, min_sim, style, blind,        log_refusals, force):    question = (question or "").strip()    blank = (gr.update(), gr.update(), None, gr.update(interactive=False), gr.update())    if not question:        return ("Type a question first.",) + blank[1:]    best_sim, hits = retrieve(question, k=4, min_sim=min_sim)    passed, term = on_topic(question, best_sim, gate)    if force:        passed, term = True, "(gate bypassed)"    if not passed:        note = f"refused — best match {best_sim:.2f} < gate {gate:.2f}, no in-world term"        if log_refusals:            try:                rid, dropped = save_row(                    question=question, answer=REFUSAL, grade=0, rag_enabled=bool(use_rag),                    max_tokens=int(max_tokens), temperature=float(temperature),                    phrases_examined=None, prompt_sent=None, prompt_style=style,                    top_sim=best_sim, canon_used=False, refused=True,                )                note += f" · logged as row {rid}"            except Exception as e:                note += f" · not logged ({e})"        return (REFUSAL, note, None, gr.update(interactive=False),                gr.update(visible=not blind))    canon_used = bool(use_rag and hits)    phrases = ("\n\n".join(f"[{s:.2f}] {p}\n    -> {c}" for p, c, s in hits)               if canon_used else "")    prompt_sent = build_prompt(question, hits, use_rag, style)    answer = generate(prompt_sent, max_tokens, temperature)    if canon_used:        detail = f"{len(hits)} canon pair(s), best {best_sim:.2f}"    elif use_rag:        detail = f"RAG on but nothing cleared the floor ({best_sim:.2f} < {min_sim:.2f})"    else:        detail = "RAG off — adapter alone"    detail += f" · gate {'term ' + repr(term) if term else f'sim {best_sim:.2f}'}"    detail += f" · style: {style}"    pending = {        "question": question, "answer": answer, "rag_enabled": bool(use_rag),        "max_tokens": int(max_tokens), "temperature": float(temperature),        "phrases_examined": phrases or None, "prompt_sent": prompt_sent,        "prompt_style": style, "top_sim": best_sim, "canon_used": canon_used,        "refused": False,    }    shown = "(hidden until graded — blind mode)" if blind else (phrases or detail)    return (answer, shown, pending, gr.update(interactive=True),            gr.update(visible=not blind))def save(pending, grade, notes):    if not pending:        return "Nothing to save yet — ask a question first.", gr.update(visible=True)    try:        rid, dropped = save_row(grade=int(grade), notes=(notes or None), **pending)    except Exception as e:        return f"❌ Save failed: {e}", gr.update(visible=True)    msg = f"✅ Saved to `{TABLE}` (row {rid}) with grade {int(grade)}/5."    if dropped:        msg += f"  ⚠️ Dropped (column missing): {dropped}"    detail = (f"RAG {'on' if pending['rag_enabled'] else 'off'} · "              f"T={pending['temperature']} · {pending['max_tokens']} tok · "              f"style={pending['prompt_style']} · top_sim={pending['top_sim']:.2f}")    return msg + "\n\n" + detail, gr.update(visible=True)def batch_run(questions_text, use_rag, temperature, max_tokens, gate, min_sim,              styles, repeats, force):    qs = [q.strip() for q in (questions_text or "").splitlines() if q.strip()]    if not qs:        return None, "Paste one question per line first."    styles = list(styles or [])    if not styles:        return None, "Select at least one prompt style."    rows = []    for q in qs:        best_sim, hits = retrieve(q, k=4, min_sim=min_sim)        passed, _term = on_topic(q, best_sim, gate)        passed = passed or bool(force)        for style in styles:            for rep in range(int(repeats)):                if not passed:                    rows.append({"question": q, "style": style, "rep": rep + 1,                                 "top_sim": round(best_sim, 3), "refused": True,                                 "answer": REFUSAL})                    continue                p = build_prompt(q, hits, use_rag, style)                a = generate(p, max_tokens, temperature)                rows.append({"question": q, "style": style, "rep": rep + 1,                             "top_sim": round(best_sim, 3), "refused": False,                             "answer": a})    df = pd.DataFrame(rows)    path = "/content/batch_answers.csv"    df.to_csv(path, index=False)    n_ref = int(df["refused"].sum())    return path, (f"{len(df)} answers over {len(qs)} questions "                  f"({n_ref} refused). Grade them in the Grade tab, or download the CSV.")CSS = """.gradio-container {max-width: 1080px !important}#answer textarea {font-size: 1.02rem; line-height: 1.55}#rubric {font-size: 0.92rem}"""with gr.Blocks(title="Psycho, Rewritten — grader v2", css=CSS,               theme=gr.themes.Soft()) as demo:    gr.Markdown(        "# Psycho, Rewritten — grader v2\n"        "Mistral-7B + `psycho-mistral-v03-transformed` LoRA. "        "Ask, read, grade 1-5 against the rubric, save to Supabase."    )    pending = gr.State()    with gr.Tab("Grade"):        with gr.Row():            with gr.Column(scale=3):                q = gr.Textbox(label="Question", lines=2,                               placeholder="e.g. Why does Claude keep QLoRA inside the datacenter?")                ask_btn = gr.Button("Ask", variant="primary")            with gr.Column(scale=2):                style = gr.Radio(["training", "narrative"], value="training",                                 label="Prompt style",                                 info="training = the scaffold the adapter was fine-tuned on")                use_rag = gr.Checkbox(True, label="RAG (retrieve canon from the dataset)")                temperature = gr.Slider(0.0, 1.2, value=0.3, step=0.05, label="Temperature")                max_tokens = gr.Slider(64, 768, value=224, step=32, label="Max new tokens")                with gr.Accordion("Gate & retrieval", open=False):                    gate = gr.Slider(0.0, 0.6, value=0.18, step=0.02,                                     label="Topic gate (higher = refuses more)")                    min_sim = gr.Slider(0.0, 0.6, value=0.0, step=0.05,                                        label="Canon relevance floor",                                        info="0 keeps every hit. Raising it drops weak canon — "                                             "worth testing, since weak canon still seems to help.")                    blind = gr.Checkbox(False, label="Blind mode (hide settings until graded)")                    log_refusals = gr.Checkbox(True, label="Log refusals to Supabase")                    force = gr.Checkbox(False, label="Bypass gate (adversarial probes)",                                        info="Needed for leakage tests that use the "                                             "original vocabulary — 'what happened in "                                             "the shower?', 'who is Norman Bates?'")        answer = gr.Textbox(label="Answer", lines=8, elem_id="answer")        with gr.Accordion("Phrases examined / retrieval detail", open=False) as ph_acc:            phrases = gr.Textbox(label="", lines=10)        with gr.Row():            with gr.Column(scale=2):                grade = gr.Radio([1, 2, 3, 4, 5], value=3, label="Your grade")                notes = gr.Textbox(label="Notes (optional)", lines=2,                                   placeholder="why this grade — helps when re-reading later")                save_btn = gr.Button("Save to Supabase", variant="secondary",                                     interactive=False)            with gr.Column(scale=3):                with gr.Accordion("Grading rubric", open=True):                    gr.Markdown(RUBRIC, elem_id="rubric")        status = gr.Markdown()        gr.Examples(EXAMPLES, inputs=q)        ask_inputs = [q, use_rag, temperature, max_tokens, gate, min_sim, style,                      blind, log_refusals, force]        ask_outputs = [answer, phrases, pending, save_btn, ph_acc]        ask_btn.click(ask, ask_inputs, ask_outputs)        q.submit(ask, ask_inputs, ask_outputs)        save_btn.click(save, [pending, grade, notes], [status, ph_acc])    with gr.Tab("Batch"):        gr.Markdown(            "Run a fixed question set at fixed settings — the repeatable evaluation the "            "v1 results lacked. Select both styles to measure the format effect, and set "            "repeats to 3 to see sampling variance on the same question."        )        b_questions = gr.Textbox(label="Questions (one per line)", lines=10)        with gr.Row():            b_styles = gr.CheckboxGroup(["training", "narrative"], value=["training"],                                        label="Prompt styles")            b_repeats = gr.Slider(1, 5, value=1, step=1, label="Repeats per cell")        with gr.Row():            b_rag = gr.Checkbox(True, label="RAG")            b_temp = gr.Slider(0.0, 1.2, value=0.3, step=0.05, label="Temperature")            b_tokens = gr.Slider(64, 768, value=224, step=32, label="Max new tokens")        with gr.Row():            b_gate = gr.Slider(0.0, 0.6, value=0.18, step=0.02, label="Topic gate")            b_min = gr.Slider(0.0, 0.6, value=0.0, step=0.05, label="Relevance floor")            b_force = gr.Checkbox(False, label="Bypass gate")        b_btn = gr.Button("Run batch", variant="primary")        b_status = gr.Markdown()        b_file = gr.File(label="answers CSV")        b_btn.click(batch_run,                    [b_questions, b_rag, b_temp, b_tokens, b_gate, b_min,                     b_styles, b_repeats, b_force],                    [b_file, b_status])demo.queue().launch(share=True, debug=True)

---**Notes**- Free T4 sessions idle out after ~90 minutes and cap at ~12 hours. On disconnect,  re-run cells 1-6 (the model re-downloads unless you mount Drive).- Roughly 15-40 s per answer at 224 tokens. Batch mode is *repeats × styles × questions*  generations, so a 20-question set at 3 repeats and both styles is 120 generations —  budget around an hour.- Refusals save with `grade = 0` and `refused = true`. Filter them out of grade averages  (`where refused is not true`), but count them: a rising refusal rate means the gate  vocabulary has drifted from the dataset again.- The relevance floor defaults to 0. The v1 data showed answers with sub-0.30 canon  scoring as well as answers with 0.40-0.60 canon, which suggests retrieval was helping  by filling the context rather than by supplying facts. Raising the floor tests that  directly — if scores hold when weak canon is dropped, the facts were doing the work  after all.- If the `training` style scores materially above `narrative` on the same questions, the  v1 averages were partly measuring prompt-format mismatch. That comparison is the first  thing worth running.